# PyTorch 基礎：Tensor、GPU 與 Autograd

<a href="https://colab.research.google.com/github/tjwei/DataScienceSoftware2026/blob/main/05-pytorch/01-PyTorch-Basics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

建議環境：Google Colab，`執行階段 → 變更執行階段類型 → T4 GPU`

In [ ]:
# 下載本章練習的參考解答 (q_*.py)
import os, urllib.request
BASE_URL = 'https://raw.githubusercontent.com/tjwei/DataScienceSoftware2026/main/05-pytorch/'
for fn in ['q_sum.py', 'q_grad_sin.py', 'q_gd.py']:
    if not os.path.exists(fn):
        urllib.request.urlretrieve(BASE_URL + fn, fn)

In [ ]:
import torch
import numpy as np
print(torch.__version__)

## 檢查 GPU

Colab 上應該會看到 `Tesla T4`。T4 是 Turing 架構 (compute capability 7.5)：
* 支援 **fp16** Tensor Core（快）
* **不支援** 原生 bf16、FlashAttention 2（這些要 Ampere 之後的卡）

所以這門課在 T4 上訓練時都使用 fp16 混合精度。

In [ ]:
!nvidia-smi

In [ ]:
print(torch.cuda.is_available())
device = "cuda" if torch.cuda.is_available() else "cpu"
if device == "cuda":
    print(torch.cuda.get_device_name(), torch.cuda.get_device_capability())

## Tensor

Tensor 基本上就是 numpy 的 ndarray，但是可以放在 GPU 上，而且可以自動微分。

In [ ]:
torch.tensor(42)

In [ ]:
torch.tensor(42.)

In [ ]:
torch.tensor([42])

In [ ]:
matrix1 = torch.tensor([[3., 3.]])
matrix2 = torch.tensor([[2.], [2.]])
matrix1, matrix2

In [ ]:
product = matrix1 @ matrix2   # 等同 torch.matmul(matrix1, matrix2)
product

In [ ]:
# 常用屬性
product.shape, product.dtype, product.device

In [ ]:
# 取出 python 數值
product.item()

### 和 numpy 互轉

In [ ]:
a = np.arange(6).reshape(2, 3)
t = torch.from_numpy(a)   # 共用記憶體
t

In [ ]:
t[0, 0] = 100
a

In [ ]:
t.numpy()

In [ ]:
# torch.tensor 則會複製一份
torch.tensor(a, dtype=torch.float32)

### 建立 tensor 的各種方式（和 numpy 幾乎一樣）

In [ ]:
torch.zeros(2, 3), torch.ones(2, 3), torch.arange(10), torch.linspace(0, 1, 5)

In [ ]:
torch.rand(2, 3), torch.randn(2, 3)

In [ ]:
x = torch.arange(12).reshape(3, 4)
x

In [ ]:
# indexing, slicing, broadcasting 都和 numpy 一樣
x[1], x[:, 1], x[x > 5], x + torch.tensor([100, 200, 300, 400])

In [ ]:
# numpy 的 axis 在 torch 叫 dim（大部分函數兩種寫法都接受）
x.sum(dim=0), x.float().mean(dim=1)

## Q
用不同的方式計算 $1+2+\cdots+10$
* 用公式 $10\cdot(10+1)/2$
* 用 `torch.sum`
* 用 `torch.cumsum`
* 用 矩陣乘法 `@`
* 用 python 迴圈

In [ ]:
# 參考解答：Jupyter 可用 %load q_sum.py；Colab 可用 !cat q_sum.py 看程式碼
%run -i q_sum.py

## GPU

用 `.to(device)` 把 tensor 搬到 GPU。參與運算的 tensor 必須在同一個 device 上。

In [ ]:
x = torch.randn(3, 3)
x_gpu = x.to(device)
x_gpu.device

In [ ]:
# 也可以直接在 GPU 上建立
y_gpu = torch.ones(3, 3, device=device)
x_gpu + y_gpu

In [ ]:
# 要轉回 numpy 必須先回到 CPU
(x_gpu + y_gpu).cpu().numpy()

### CPU vs GPU 速度比較

注意 GPU 運算是**非同步**的，計時前後要呼叫 `torch.cuda.synchronize()`。

In [ ]:
import time
def benchmark(device, dtype=torch.float32, n=4096, repeat=10):
    a = torch.randn(n, n, device=device, dtype=dtype)
    b = torch.randn(n, n, device=device, dtype=dtype)
    a @ b  # warm up
    if device == "cuda": torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(repeat):
        c = a @ b
    if device == "cuda": torch.cuda.synchronize()
    dt = (time.perf_counter() - t0) / repeat
    print(f"{device} {dtype}: {dt*1000:.1f} ms,  {2*n**3/dt/1e12:.2f} TFLOPS")

benchmark("cpu", repeat=2)
if device == "cuda":
    benchmark("cuda", torch.float32)
    benchmark("cuda", torch.float16)   # T4 的 Tensor Core 在 fp16 才有威力

## Autograd 自動微分

設定 `requires_grad=True`，PyTorch 會記錄運算過程（動態計算圖），呼叫 `.backward()` 就能算出梯度。

In [ ]:
x = torch.tensor(3.0, requires_grad=True)
y = x**2 + 2*x + 1
y

In [ ]:
y.backward()
x.grad    # dy/dx = 2x + 2 = 8

In [ ]:
# 多變數
w = torch.tensor([1., 2., 3.], requires_grad=True)
loss = (w**2).sum()
loss.backward()
w.grad

注意：梯度會**累加**，每次重新計算前要清掉。

In [ ]:
loss = (w**2).sum()
loss.backward()
w.grad   # 變成兩倍了

In [ ]:
w.grad.zero_()
w.grad

不需要梯度時（例如推論 inference），用 `torch.no_grad()` 或 `torch.inference_mode()` 可以省記憶體、加速。

In [ ]:
with torch.no_grad():
    z = w * 2
z.requires_grad

In [ ]:
# .detach() 取得一個不追蹤梯度的 tensor
w.detach()

## Q
用 autograd 計算 $\sin(x)$ 在 $x\in[0, 2\pi]$ 上 100 個點的導數，並和 $\cos(x)$ 比較，畫出圖來。

提示：對一個向量 `y`，可以用 `y.sum().backward()`。

In [ ]:
# %load q_grad_sin.py
%run -i q_grad_sin.py

## Gradient descent

有了梯度，就可以用梯度下降法找最小值。

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
lr = 0.1
for i in range(30):
    y = (x - 2)**2
    y.backward()
    with torch.no_grad():
        x -= lr * x.grad
    x.grad.zero_()
    if i % 5 == 0:
        print(i, x.item(), y.item())

上面手動更新參數的部份，PyTorch 已經幫我們寫好了，就是 `torch.optim`。

In [ ]:
x = torch.tensor(5.0, requires_grad=True)
optimizer = torch.optim.SGD([x], lr=0.1)
for i in range(30):
    optimizer.zero_grad()
    y = (x - 2)**2
    y.backward()
    optimizer.step()
x

## Q
用 gradient descent 找出 Rosenbrock function
$$f(x, y) = (1-x)^2 + 100(y-x^2)^2$$
的最小值（答案是 $(1,1)$）。
* 從 $(-1, 2)$ 出發
* 比較 `torch.optim.SGD` 和 `torch.optim.Adam`
* 試試看不同的 learning rate

In [ ]:
# %load q_gd.py
%run -i q_gd.py